In [ ]:
# ! pip install selenium


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time

In [3]:


def crawl_company_profile(url):
    print(f"🤖 Robot đang truy cập: {url}")
    
    # --- GIAI ĐOẠN 1: SELENIUM (NGƯỜI VẬN CHUYỂN) ---
    # Cấu hình Chrome ẩn (Headless) để chạy ngầm cho nhanh
    chrome_options = Options()
    # chrome_options.add_argument("--headless") # Bỏ comment dòng này nếu muốn ẩn trình duyệt
    
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        driver.get(url)
        
        # MẤU CHỐT: Đợi 5 giây để JavaScript vẽ xong các biểu đồ và bảng
        # Với dân pro thì dùng WebDriverWait, nhưng với người mới: time.sleep là dễ hiểu nhất
        print("⏳ Đang đợi trang web tải dữ liệu (5s)...")
        time.sleep(5) 
        
        # Lấy toàn bộ mã nguồn HTML sau khi đã render xong
        html_source = driver.page_source
        
    finally:
        driver.quit() # Nhớ tắt trình duyệt để không tốn RAM
        print("✅ Đã lấy xong HTML, bắt đầu mổ xẻ...")

    # --- GIAI ĐOẠN 2: BEAUTIFULSOUP (BÁC SĨ PHẪU THUẬT) ---
    soup = BeautifulSoup(html_source, 'html.parser')
    
    # 1. BÓC TÁCH CHỈ SỐ CƠ BẢN (P/E, EPS...)
    # Logic: Tìm class "chi-so-item", bên trong có "name" và "value"
    indicators = {}
    items = soup.find_all('div', class_='chi-so-item')
    
    for item in items:
        name = item.find('span', class_='chi-so-item-name').text.strip()
        value = item.find('span', class_='chi-so-item-value').text.strip()
        indicators[name] = value
        
    df_indicators = pd.DataFrame([indicators]) # Chuyển thành DataFrame 1 dòng
    
    # 2. BÓC TÁCH CƠ CẤU CỔ ĐÔNG
    # Logic: Tìm class "ownership-dashboard__item"
    owners = []
    owner_items = soup.find_all('div', class_='ownership-dashboard__item')
    
    for item in owner_items:
        # Lấy tên cổ đông (Class 'ownership-dashboard__label' chứa cả tên và tỷ lệ)
        full_text = item.find('span', class_='ownership-dashboard__label').text.strip()
        
        # Tách tên và tỷ lệ (Mẹo xử lý chuỗi đơn giản)
        # Ví dụ: "Tập đoàn... (55.06%)" -> Tách theo dấu mở ngoặc
        if '(' in full_text:
            name = full_text.split('(')[0].strip()
            percent = full_text.split('(')[1].replace('%)', '').strip()
        else:
            name = full_text
            percent = 0
            
        owners.append({'Cổ đông': name, 'Tỷ lệ (%)': percent})
        
    df_owners = pd.DataFrame(owners)
    
    # 3. BÓC TÁCH SO SÁNH CÙNG NGÀNH
    # Logic: Tìm bảng có class "tablePrice". Pandas đọc bảng HTML cực giỏi.
    try:
        # Chuyển đối tượng soup thành chuỗi để pandas đọc
        table_node = soup.find('table', class_='tablePrice')
        if table_node:
            # pd.read_html trả về 1 list các bảng, ta lấy bảng đầu tiên [0]
            df_peers = pd.read_html(str(table_node))[0]
        else:
            df_peers = pd.DataFrame()
    except Exception as e:
        print(f"Lỗi đọc bảng so sánh: {e}")
        df_peers = pd.DataFrame()

    return df_indicators, df_owners, df_peers


In [ ]:

# --- CHẠY THỬ NGHIỆM ---
url_target = "https://m.cafef.vn/du-lieu/hose/hrc-cong-ty-co-phan-cao-su-hoa-binh.chn"
df_ind, df_own, df_peer = crawl_company_profile(url_target)

print("\n=== 1. CHỈ SỐ CƠ BẢN ===")
print(df_ind.T) # Transpose (xoay dọc) cho dễ đọc

print("\n=== 2. CƠ CẤU CỔ ĐÔNG ===")
print(df_own)

print("\n=== 3. SO SÁNH CÙNG NGÀNH ===")
print(df_peer[['Mã', 'EPS', 'PE', 'Giá']].head())

🤖 Robot đang truy cập: https://m.cafef.vn/du-lieu/hose/hrc-cong-ty-co-phan-cao-su-hoa-binh.chn


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import random # Thêm cái này để random thời gian nghỉ

def crawl_company_profile_stealth(url):
    print(f"🕵️‍♂️ Robot (Tàng hình) đang truy cập: {url}")
    
    # --- CẤU HÌNH NGỤY TRANG (QUAN TRỌNG) ---
    chrome_options = Options()
    
    # 1. Giả mạo User-Agent (Như người dùng thật trên Window 10)
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # 2. Tắt tính năng phát hiện Automation của Chrome
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    
    # 3. Ẩn thanh thông báo "Chrome is being controlled..."
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)

    # Khởi tạo driver với cấu hình mới
    driver = webdriver.Chrome(options=chrome_options)
    
    # Mẹo nâng cao: Thay đổi giá trị navigator.webdriver bằng JavaScript để chắc chắn không bị lộ
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    try:
        driver.get(url)
        
        # Random thời gian đợi từ 5-7 giây cho giống người
        sleep_time = random.uniform(5, 7)
        print(f"⏳ Đang đợi trang web tải dữ liệu ({sleep_time:.2f}s)...")
        time.sleep(sleep_time) 
        
        # Kiểm tra xem có bị chặn không bằng cách in Title
        print(f"Title trang web: {driver.title}")
        
        html_source = driver.page_source
        
    finally:
        driver.quit()
        print("✅ Đã lấy xong HTML.")

    # --- PHẦN BÓC TÁCH (GIỮ NGUYÊN NHƯ CŨ) ---
    soup = BeautifulSoup(html_source, 'html.parser')
    
    # (Đoạn code parsing giữ nguyên như bài trước...)
    # Copy lại phần parsing indicator, owners, peers vào đây
    
    # ... Ví dụ test nhanh phần tiêu đề để xem có load được không
    try:
        # Thử lấy giá xem được chưa
        price = soup.find('span', class_='price') # Class này có thể thay đổi tùy giao diện mobile
        if price:
            print(f"Giá tìm thấy: {price.text}")
    except:
        pass

    return soup # Trả về soup để debug xem lấy được gì



In [ ]:
# --- CHẠY THỬ ---
url_target = "https://m.cafef.vn/du-lieu/hose/hrc-cong-ty-co-phan-cao-su-hoa-binh.chn"
soup_result = crawl_company_profile_stealth(url_target)

# Kiểm tra thử xem lấy được bảng chỉ số chưa
items = soup_result.find_all('div', class_='chi-so-item')
print(f"\nTìm thấy {len(items)} chỉ số cơ bản.")
if len(items) > 0:
    print("Ví dụ:", items[0].text.strip())
else:
    print("⚠️ Vẫn chưa lấy được dữ liệu. Có thể web chặn IP hoặc cấu trúc thay đổi.")

🕵️‍♂️ Robot (Tàng hình) đang truy cập: https://m.cafef.vn/du-lieu/hose/hrc-cong-ty-co-phan-cao-su-hoa-binh.chn
⏳ Đang đợi trang web tải dữ liệu (6.84s)...
Title trang web: HRC: Công ty Cổ phần Cao su Hòa Bình (HOSE)
✅ Đã lấy xong HTML.

Tìm thấy 0 chỉ số cơ bản.
⚠️ Vẫn chưa lấy được dữ liệu. Có thể web chặn IP hoặc cấu trúc thay đổi.


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import os

# Hàm ghi file HTML an toàn với tiếng Việt
def save_html(content, filename):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

def debug_investigation(url):
    print("🕵️‍♂️ BẮT ĐẦU ĐIỀU TRA HIỆN TRƯỜNG...")
    print(f"🔗 Target: {url}")
    
    # 1. CẤU HÌNH NGỤY TRANG (STEALTH MODE)
    # Phải có đoạn này để tránh bị chặn ngay từ cửa
    chrome_options = Options()
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)

    # Khởi tạo trình duyệt (Có hiện giao diện để bạn nhìn thấy process)
    driver = webdriver.Chrome(options=chrome_options)

    try:
        # 2. TRUY CẬP
        driver.get(url)
        
        print("⏳ Đang đợi trang web load (10s)...")
        time.sleep(10) # Đợi lâu một chút cho chắc ăn
        
        # In ra tiêu đề trang để xem có vào đúng không
        print(f"✅ Page Title nhận được: {driver.title}")

        # 3. LƯU HIỆN TRƯỜNG (QUAN TRỌNG NHẤT)
        
        # A. Chụp ảnh màn hình
        img_filename = "debug_screenshot.png"
        driver.save_screenshot(img_filename)
        print(f"📸 Đã lưu ảnh màn hình: {os.path.abspath(img_filename)}")
        
        # B. Lưu Source HTML
        html_filename = "debug_source.html"
        page_source = driver.page_source
        save_html(page_source, html_filename)
        print(f"💾 Đã lưu file HTML: {os.path.abspath(html_filename)}")
        
        # 4. PHÂN TÍCH NHANH (QUICK SCAN)
        soup = BeautifulSoup(page_source, 'html.parser')
        
        # Thử tìm từ khóa "P/E" xem nó có tồn tại trong HTML không
        keyword = "P/E"
        if keyword in soup.get_text():
            print(f"✅ TÌM THẤY từ khóa '{keyword}' trong nội dung trang web!")
        else:
            print(f"❌ KHÔNG THẤY từ khóa '{keyword}'. Trang web có thể bị trắng hoặc chặn.")

        # Thử tìm class cũ xem có không
        class_check = soup.find_all('div', class_='chi-so-item')
        print(f"🔍 Kiểm tra class 'chi-so-item': Tìm thấy {len(class_check)} phần tử.")

    except Exception as e:
        print(f"⚠️ CÓ LỖI XẢY RA: {e}")

    finally:
        driver.quit()
        print("👮‍♂️ KẾT THÚC ĐIỀU TRA. Hãy mở file HTML và PNG để kiểm tra.")

# --- CHẠY THỬ ---
# Link Mobile (m.cafef.vn)
url_target = "https://m.cafef.vn/du-lieu/hose/hrc-cong-ty-co-phan-cao-su-hoa-binh.chn"
debug_investigation(url_target)

🕵️‍♂️ BẮT ĐẦU ĐIỀU TRA HIỆN TRƯỜNG...
🔗 Target: https://m.cafef.vn/du-lieu/hose/hrc-cong-ty-co-phan-cao-su-hoa-binh.chn
⏳ Đang đợi trang web load (10s)...
✅ Page Title nhận được: HRC: Công ty Cổ phần Cao su Hòa Bình (HOSE)
📸 Đã lưu ảnh màn hình: /home/kyhoolee/work/9_teaching/python_course/module9/debug_screenshot.png
💾 Đã lưu file HTML: /home/kyhoolee/work/9_teaching/python_course/module9/debug_source.html
✅ TÌM THẤY từ khóa 'P/E' trong nội dung trang web!
🔍 Kiểm tra class 'chi-so-item': Tìm thấy 0 phần tử.
👮‍♂️ KẾT THÚC ĐIỀU TRA. Hãy mở file HTML và PNG để kiểm tra.


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import io

# Giả sử đây là HTML bạn lấy được từ Selenium driver.page_source
# (Mình ghép 2 đoạn HTML bạn gửi vào một biến để demo)
html_content = """
<html>
<body>
    <table class="report_detail_table">
        <thead>
            <tr>
                <th colspan="2">Chỉ tiêu</th>
                <th></th> <th>Q1-2025</th><th>Q2-2025</th><th>Q3-2025</th><th>Q4-2025</th> <th></th> <th>Tăng trưởng</th>
            </tr>
        </thead>
        <tbody>
            <tr class="btn-even">
                <td colspan="2">Doanh thu bán hàng và CCDV</td>
                <td></td>
                <td>37.35 tỷ</td><td>37.35 tỷ</td><td>59.46 tỷ</td><td>67.23 tỷ</td>
            </tr>
            <tr class="btn-odd">
                 <td colspan="2">Lợi nhuận sau thuế</td>
                 <td></td>
                 <td>1.43 tỷ</td><td>1.43 tỷ</td><td>3.32 tỷ</td><td>22.04 tỷ</td>
            </tr>
        </tbody>
    </table>

    <div class="table-right" id="transaction-information-table-right">
        <div class="table-right-item">
            <p>EPS cơ bản <span>(nghìn đồng)</span></p>
            <p>0.93</p>
        </div>
        <div class="table-right-item">
            <p>P/E</p>
            <p>30.45</p>
        </div>
        <div class="table-right-item">
            <p>Vốn hóa thị trường <span>(tỷ đồng)</span></p>
            <p>859.38</p>
        </div>
    </div>
</body>
</html>
"""

def parse_cafef_desktop(html_source):
    soup = BeautifulSoup(html_source, 'html.parser')
    
    # --- PHẦN 1: BÓC TÁCH BẢNG KẾT QUẢ KINH DOANH (TABLE) ---
    print("📊 Đang xử lý Bảng Kết quả kinh doanh...")
    
    # Tìm bảng theo class
    table = soup.find('table', class_='report_detail_table')
    
    financial_data = []
    
    if table:
        # Lặp qua từng dòng dữ liệu (tr)
        # Bỏ qua dòng header, chỉ lấy dòng có class btn-even hoặc btn-odd
        rows = table.find_all('tr')
        
        for row in rows:
            # Chỉ xử lý những dòng có dữ liệu (thường có class btn-...)
            # Hoặc kiểm tra xem có đủ số lượng thẻ td không
            cols = row.find_all('td')
            
            if len(cols) > 3: # Đảm bảo dòng có dữ liệu
                # Cấu trúc HTML của CafeF:
                # [0]: Tên chỉ tiêu (colspan=2)
                # [1]: Nút Prev (Rỗng) -> Bỏ qua
                # [2], [3], [4], [5]: Dữ liệu 4 quý
                
                indicator_name = cols[0].text.strip()
                
                # Lưu ý: Index trong cols sẽ phụ thuộc vào số lượng quý hiển thị
                # Ta dùng List Comprehension để lấy text của các ô còn lại, bỏ qua ô rỗng
                values = [ele.text.strip() for ele in cols[1:] if ele.text.strip() != '']
                
                # Đưa vào danh sách
                row_dict = {'Chỉ tiêu': indicator_name}
                
                # Gán nhãn Quý (Demo gán cứng, thực tế có thể lấy từ Header)
                # Giả sử lấy 4 giá trị cuối cùng tìm được làm 4 quý
                if len(values) >= 4:
                    row_dict['Quý 1'] = values[-4]
                    row_dict['Quý 2'] = values[-3]
                    row_dict['Quý 3'] = values[-2]
                    row_dict['Quý 4'] = values[-1]
                
                financial_data.append(row_dict)
    
    df_finance = pd.DataFrame(financial_data)

    # --- PHẦN 2: BÓC TÁCH CHỈ SỐ CƠ BẢN (DIV) ---
    print("📈 Đang xử lý Chỉ số cơ bản (EPS, P/E)...")
    
    basic_data = {}
    
    # Tìm container chứa các chỉ số
    container = soup.find('div', id='transaction-information-table-right')
    
    if container:
        # Mỗi chỉ số nằm trong 1 div class="table-right-item"
        items = container.find_all('div', class_='table-right-item')
        
        for item in items:
            # Mỗi item có 2 thẻ <p>: 
            # <p> thứ nhất là Tên (Key)
            # <p> thứ hai là Giá trị (Value)
            p_tags = item.find_all('p')
            
            if len(p_tags) >= 2:
                # Xử lý Tên: Thường có thêm span đơn vị (nghìn đồng), ta có thể bỏ hoặc giữ
                key = p_tags[0].text.strip()
                value = p_tags[1].text.strip()
                
                basic_data[key] = value

    # Chuyển dictionary thành DataFrame (xoay dọc cho dễ nhìn)
    df_basic = pd.DataFrame(list(basic_data.items()), columns=['Chỉ số', 'Giá trị'])

    return df_finance, df_basic

# --- CHẠY THỬ VỚI ĐOẠN HTML BẠN CUNG CẤP ---
# (Trong thực tế bạn sẽ truyền driver.page_source vào hàm này)
# Để test code này, bạn có thể copy đoạn HTML mẫu ở trên
df_fin, df_bas = parse_cafef_desktop(html_content)

print("\n=== KẾT QUẢ 1: BÁO CÁO TÀI CHÍNH ===")
print(df_fin)

print("\n=== KẾT QUẢ 2: CHỈ SỐ ĐỊNH GIÁ ===")
print(df_bas)

📊 Đang xử lý Bảng Kết quả kinh doanh...
📈 Đang xử lý Chỉ số cơ bản (EPS, P/E)...

=== KẾT QUẢ 1: BÁO CÁO TÀI CHÍNH ===
                     Chỉ tiêu     Quý 1     Quý 2     Quý 3     Quý 4
0  Doanh thu bán hàng và CCDV  37.35 tỷ  37.35 tỷ  59.46 tỷ  67.23 tỷ
1          Lợi nhuận sau thuế   1.43 tỷ   1.43 tỷ   3.32 tỷ  22.04 tỷ

=== KẾT QUẢ 2: CHỈ SỐ ĐỊNH GIÁ ===
                         Chỉ số Giá trị
0       EPS cơ bản (nghìn đồng)    0.93
1                           P/E   30.45
2  Vốn hóa thị trường (tỷ đồng)  859.38
